In [ ]:
# """
# ingestion_full_8KB.py  —  VerdaSense KB Ingestion Pipeline (db_wound_care_v4)
# ═══════════════════════════════════════════════════════════════════════════════
# Ingests all 8 curated _kept.json sources into ChromaDB (db_wound_care_v4/).

# CHANGES vs ingestion_full.ipynb (db_wound_care_v3 — 4 sources only):
# ───────────────────────────────────────────────────────────────────────
# [NEW 1]  4 new sources added: EWMA, ISTAP, ANZBA, RCH
#          → Total chunks: 108 (v3) → ~138 (v4, +28%)
# [NEW 2]  wound_type metadata field — added to every chunk
#          Enables Sub-query A metadata filter in wound_app_v5.py to pin
#          algorithm chunks ({"wound_type": {"$eq": "1"}}).
#          Value: "1"–"8" (GP wound types), "burn", "skin_tear",
#                 "dfu", "vlu", "general", "paediatric", "procedure"
# [NEW 3]  wound_category metadata field — added to every chunk
#          Enables Sub-query B to filter by dressing category.
#          Value: "algorithm", "dressing_product", "dressing_mechanism",
#                 "assessment", "procedure", "reference"
# [NEW 4]  population metadata field — normalised across all sources
#          Value: "adult" (default) | "paediatric" (RCH only)
#          Enables Sub-query B population filter so RCH paediatric chunks
#          are not retrieved for adult queries.
# [NEW 5]  GUIDELINE_METADATA entries for all 4 new sources (EWMA, ISTAP,
#          ANZBA, RCH) — provides authority, year, guideline_type for
#          generation prompt source labels.
# [FIX]    Embedding done on ai_summary (page_content) with raw text stored
#          in metadata["raw_text"] — unchanged from v3 design.

# CELL-BY-CELL STRUCTURE (convert to .ipynb cells at the | marks):
# │ Cell 1  — Imports & config
# │ Cell 2  — Embedding model
# │ Cell 3  — GUIDELINE_METADATA (all 8 sources)
# │ Cell 4  — WOUND_TYPE_MAP + WOUND_CATEGORY_MAP + POPULATION_MAP
# │ Cell 5  — Load all 8 JSON files
# │ Cell 6  — chunks_to_documents() converter
# │ Cell 7  — Ingest into ChromaDB
# │ Cell 8  — Verification & summary

# Usage:
#     python ingestion_full_8KB.py
#     # Or convert each cell block to a notebook cell and run in Jupyter.

# Requirements:
#     pip install langchain-chroma langchain-huggingface chromadb
#                 sentence-transformers torch
# """

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 1 — Imports & Configuration
# ══════════════════════════════════════════════════════════════════════════════

import os
import json
import re
import shutil
import torch
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document as LC_Doc

# ── Paths ─────────────────────────────────────────────────────────────────────
# CHUNK_DIR: folder containing all 8 _kept.json files
# DB_DIR:    output ChromaDB directory (do NOT point to v3 — separate directory)

CHUNK_DIR = os.environ.get(
    "WOUND_CHUNK_DIR",
    "ingestion_output_ai",       # ← change to your local path
)
DB_DIR = os.environ.get(
    "WOUND_DB_DIR",
    r"db_wound_care_v4",         # ← change to your local path
)

COLLECTION_NAME = "wound_care_v4"
BATCH_SIZE      = 32             # ChromaDB upsert batch size

print(f"CHUNK_DIR : {CHUNK_DIR}")
print(f"DB_DIR    : {DB_DIR}")
print(f"torch GPU : {torch.cuda.is_available()}")

c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CHUNK_DIR : ingestion_output_ai
DB_DIR    : db_wound_care_v4
torch GPU : True


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 2 — Embedding Model
# ══════════════════════════════════════════════════════════════════════════════

print("\n[Cell 2] Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="abhinand/MedEmbed-large-v0.1",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("  ✅ Embedding model loaded.")



[Cell 2] Loading embedding model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3716.26it/s]


  ✅ Embedding model loaded.


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 3 — GUIDELINE_METADATA (all 8 sources)
# ══════════════════════════════════════════════════════════════════════════════

# Keyed by the value that appears in chunk["source"] field (or a prefix match).
# Used to enrich each document's metadata so the generation prompt can produce
# proper source labels, e.g. "GP Wound Care Guideline [MOH Malaysia, 2023]".

GUIDELINE_METADATA = {
    # ── Original 4 sources (v3) ───────────────────────────────────────────────
    "GP_wound_dressings": {
        "authority":      "Ministry of Health Malaysia (MOH)",
        "year":           "2019",
        "guideline_type": "national_guideline",
        "full_name":      "Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer",
        "abbreviation":   "GP",
    },
    "WCM": {
        "authority":      "Ministry of Health Malaysia (MOH)",
        "year":           "2014",
        "guideline_type": "clinical_manual",
        "full_name":      "Wound Care Manual - First Edition",
        "abbreviation":   "WCM",
    },
    "AJGP": {
        "authority":      "Australian Journal of General Practice",
        "year":           "2022",
        "guideline_type": "clinical_review",
        "full_name":      "The art and science of selecting appropriate dressings for acute open wounds in general practice",
        "abbreviation":   "AJGP",
    },
    "SFP": {
        "authority":      "Singapore Family Physician",
        "year":           "2018",
        "guideline_type": "clinical_education",
        "full_name":      "Wound Dressings - A Primer For The Family Physician",
        "abbreviation":   "SFP",
    },
    # ── New 4 sources (v4) ────────────────────────────────────────────────────
    "EWMA": {
        "authority":      "European Wound Management Association (EWMA)",
        "year":           "2004",
        "guideline_type": "international_evidence_based_position_document",
        "full_name":      "Wound Bed Preparation In Practice",
        "abbreviation":   "EWMA",
    },
    "ISTAP": {
        "authority":      "International Skin Tear Advisory Panel (ISTAP)",
        "year":           "2024",
        "guideline_type": "international_consensus_guideline",
        "full_name":      "Skin Tear Classification and Management (ISTAP)",
        "abbreviation":   "ISTAP",
    },
    "ANZBA": {
        "authority":      "Australia and New Zealand Burn Association (ANZBA)",
        "year":           "2021",
        "guideline_type": "national_specialist_guideline",
        "full_name":      "Burns Assessment and Management Guidelines (ANZBA)",
        "abbreviation":   "ANZBA",
    },
    "RCH": {
        "authority":      "Royal Children's Hospital Melbourne (RCH)",
        "year":           "2023",
        "guideline_type": "paediatric_clinical_practice_guideline",
        "full_name":      "Wound Assessment and Management — RCH Nursing Guideline (2023)",
        "abbreviation":   "RCH",
    },
}


def _resolve_guideline_meta(chunk: dict) -> dict:
    """
    Match a chunk's source field to GUIDELINE_METADATA.
    Uses prefix matching so 'GP_wound_dressings.pdf' matches 'GP_wound_dressings'.
    Falls back to empty dict if no match (never crashes).
    """
    src = chunk.get("source", "")
    for key, meta in GUIDELINE_METADATA.items():
        if key.lower() in src.lower():
            return meta
    # Secondary: match by abbreviation in source string
    for key, meta in GUIDELINE_METADATA.items():
        if meta["abbreviation"].lower() in src.lower():
            return meta
    return {}


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 4 — wound_type, wound_category, population Maps
# ══════════════════════════════════════════════════════════════════════════════

# ── wound_type: chunk_id → wound_type string ──────────────────────────────────
# "1"–"8"    : MOH GP wound types (directly maps to Sub-query A filter)
# "burn"     : burn wound (ANZBA, WCM burn chapter, AJGP burns table)
# "skin_tear": skin tear (ISTAP, AJGP skin tear table)
# "dfu"      : diabetic foot ulcer (EWMA DFU, WCM Ch8a, AJGP diabfoot)
# "vlu"      : venous leg ulcer (EWMA VLU, WCM Ch8b)
# "general"  : cross-cutting / TIME framework / dressing reference
# "paediatric": RCH paediatric-specific (overridden by population field)
# "procedure": procedural chapters (debridement, NPWT, honey, etc.)

WOUND_TYPE_MAP: dict[str, str] = {

    # ── GP: algorithm + wound types 1–8 ──────────────────────────────────────
    "8409dedeea26": "general",    # GP TIME Framework
    "bd2bb8e1321e": "general",    # GP Decision Algorithm  ← the pinned algo chunk
    "52ef696853c7": "1",          # GP Wound Type 1
    "4643f10b8894": "2",          # GP Wound Type 2
    "c0a350e36ecf": "3",          # GP Wound Type 3
    "d622ee9f4c9c": "4",          # GP Wound Type 4
    "aad7a40107b0": "5",          # GP Wound Type 5
    "b4ba04cb08d4": "6",          # GP Wound Type 6
    "c4177e98524e": "7",          # GP Wound Type 7
    "e75347f9bdb3": "8",          # GP Wound Type 8
    "ca7a1e934891": "general",    # GP Referral criteria
    "8bb5168bd7e5": "general",    # GP Tissue type descriptions
    "0cc16688a29a": "general",    # GP Clinical glossary

    # ── SFP (36 chunks) ───────────────────────────────────────────────────────
    "ba05f42e79d0": "general",    # Abstract
    "08021aac2fa6": "general",    # Wound Dressings Selection Factors
    "e7ca1b602a48": "general",    # Wound Dressings Selection Factors
    "18ab673003dc": "general",    # Wound Dressings Selection Factors
    "07da3fcbedeb": "general",    # Categories of Wound Dressings overview
    "14535d1dba1a": "general",    # Moisture Retentive Dressings
    "8443058d681e": "general",    # Absorbent Dressings
    "c42103c45a0c": "general",    # Absorbent Dressings
    "80e829def300": "general",    # Antimicrobial Dressings
    "3082e1a296e7": "general",    # Antimicrobial Dressings (iodine/silver)
    "5972b0ef4b66": "general",    # Composite Dressings
    "b49bc7f134bd": "general",    # Protective Dressings
    "08bd7cc4c34e": "general",    # Advances in Wound Care Technology
    "9c6c8076a17e": "procedure",  # Maggot Debridement Therapy
    "138be9123963": "general",    # Growth Factors
    "218879320005": "general",    # Growth Factors
    "52981701d491": "general",    # Growth Factors
    "8b8a70dc7e9e": "general",    # Bioengineered Skin Substitutes
    "ea9194262a27": "procedure",  # NPWT
    "1fddeefdfb8b": "procedure",    # Oxygen Therapy HBOT
    "a23f76944381": "procedure",    # Ultrasound Therapy
    "352258a65d26": "procedure",    # Low-Power Laser Therapy
    "8ac10c306384": "general",    # Conclusions
    "25a2969b1f2e": "general",    # Learning Points
    "6a4054e5b255": "general",    # Learning Points
    "b4c13d77818b": "general",    # Table 2 Hydrocolloids
    "ad036dc35955": "general",    # Table 2 Hydrogels
    "3b666ccfba99": "general",    # Table 2 Alginates
    "4b75fefc0517": "general",    # Table 2 Hydrofiber
    "254dd74d7f00": "general",    # Table 2 Foams
    "f05456b64b8d": "general",    # Table 2 Cadexomer Iodine
    "765445bd6358": "general",    # Table 2 Silver Barrier
    "757123a126d6": "general",    # Table 2 Non-Adherent Synthetic
    "9e661711e520": "general",    # Table 2 Films/Membranes
    "605598290337": "general",    # Table 2 Gauze
    "e86bb5a0ee36": "general",    # Table 2 Composite Dressing

    # ── AJGP (19 chunks) ──────────────────────────────────────────────────────
    "65414974ff6f": "general",    # Background & Objective
    "e33a690e240a": "general",    # Introduction Acute Wound Context
    "2dc1f26b6233": "general",    # Wound Dressings General Principles
    "4b741bf1a512": "general",    # Wound Dressings General Principles
    "6e34c2b50050": "general",    # Description of Dressings Overview
    "55569f16010f": "skin_tear",  # Table 1 Skin tears
    "18fdcc9d34d9": "general",    # Table 1 Minor cut/laceration
    "6555b7728ccc": "general",    # Table 1 Postoperative wounds
    "5116098922da": "burn",       # Table 1 Small superficial burns
    "0d0a9fc09c73": "dfu",        # Table 1 Diabetic foot
    "f7b66b783d4d": "general",    # Dressing Types Film
    "c9b68759e2a8": "general",    # Dressing Types Foam
    "e5409b86f782": "general",    # Dressing Types Low-Adherent
    "a1968343e3ba": "general",    # Dressing Types Hydrocolloid
    "38f855618fe2": "general",    # Dressing Types Alginate
    "0c49aa8d13c4": "general",    # Dressing Types Antimicrobial
    "ca472c44f89b": "general",    # Dressing Types and Costs
    "751c661d8c3c": "general",    # Conclusion
    "10c6b59b1aff": "general",    # Key Points

    # ── WCM (40 chunks) ───────────────────────────────────────────────────────
    "0305af9f4828": "general",    # Ch1 Skin Anatomy
    "06f9b3a7dc69": "general",    # Ch2 Wound Definition & Classification
    "9f8aabde769e": "general",    # Ch3 Wound Assessment Principles
    "9eace8705738": "general",    # Ch4 Wound Infection Pathway
    "424af37cbc2e": "general",    # Ch4 Antibiotic Treatment
    "5c0355fe4e4c": "burn",       # Ch7a Burn Wound Assessment & Management
    "594eb6407bf0": "general",    # Ch7b Traumatic Wound
    "e21d5fa201a5": "dfu",        # Ch8a DFU Assessment & Wagner Classification
    "261f5855cdb7": "dfu",        # Ch8a DFU Management & Foot Care
    "4b89a41a249d": "vlu",        # Ch8b Venous Ulcer
    "90610861be44": "general",    # Ch8c Arterial Ulcer
    "cde2bed226ee": "general",    # Ch8d Pressure Ulcer
    "7ef21910c0c1": "general",    # Ch9 Non-Healing Ulcer
    "c6bd9ad78883": "general",    # Ch9 Non-Healing Ulcer
    "3ab46ddb1cb9": "general",    # Ch10 Necrotizing Fasciitis
    "634395d03254": "general",    # Ch10 Necrotizing Fasciitis
    "1101f1aec597": "procedure",  # Ch11 Pain Management
    "42616730845e": "procedure",  # Ch11 Pain Management
    "d1b94d1b0458": "procedure",  # Ch11 Pain Management
    "9177c6a0a15f": "procedure",  # Ch11 Pain Management
    "fd2fb7bfa667": "procedure",  # Ch11 Pain Management
    "fc529d1bd75b": "procedure",  # Ch13 Wound Cleansing Solutions
    "1b2820149c46": "general",    # Ch14 Dressing Purpose & Categories Overview
    "2de03f803f2f": "general",    # Ch14 Film
    "d81176511903": "general",    # Ch14 Hydrogel
    "f8cb463d04cf": "general",    # Ch14 Hydrocolloid
    "c540b3e5c067": "general",    # Ch14 Calcium Alginate
    "77e6e32d188a": "general",    # Ch14 Foams
    "e63bd0378895": "general",    # Ch14 Hydrofibre
    "861a57a2172c": "general",    # Ch14 Charcoal
    "e8c86c4e1aa6": "general",    # Ch14 Silver
    "6fd9e2433cc9": "general",    # Ch14 Polymeric Membrane
    "9466f8d45d86": "general",    # Ch14 Composite Dressing
    "ac13950bf23d": "general",    # Ch14 Other Advanced Dressings
    "b5b5a6c9dcf2": "procedure",  # Ch15 Wound Debridement Methods
    "c550f2c4e065": "procedure",  # Ch15 Wound Debridement Methods
    "c45690235ea6": "procedure",  # Ch15 Wound Debridement Methods
    "b480aa73a9c2": "general",    # Ch16a Honey Dressing
    "05cc6ca1ddfc": "procedure",  # Ch16c NPWT
    "c799dd10dcf3": "procedure",  # Appendix 7 Analgesics

    # ── EWMA (12 chunks) ──────────────────────────────────────────────────────
    "34a1a51c74ae": "general",    # TIME Framework — Evolution & Four Components
    "e905d7d38dad": "general",    # TIME Applied to Practice — Pathway & WBP
    "643cd131813b": "general",    # TIME Figure 1 — Dynamic Wound Progression
    "61346aa382bd": "dfu",        # DFU — Before TIME & Tissue Management
    "65a59d1430fd": "dfu",        # DFU — Inflammation & Infection Control
    "9525c0cb50e8": "dfu",        # DFU — Moisture Balance
    "6231fe39e6d5": "dfu",        # DFU — Edge Advancement
    "dfb568f55887": "dfu",        # DFU — Advanced Therapies & After TIME
    "142adfaa2033": "vlu",        # VLU — Before TIME & Tissue Management
    "bd87881f1796": "vlu",        # VLU — Inflammation & Infection Control
    "9d379b10e0c1": "vlu",        # VLU — Moisture Balance
    "a60e6a06f137": "vlu",        # VLU — Edge Advancement & Advanced Therapies

    # ── ISTAP (3 chunks) ──────────────────────────────────────────────────────
    "d2957f5e4841": "skin_tear",  # ISTAP Classification Types 1, 2, 3
    "c3d5e1f498ba": "skin_tear",  # ISTAP Pathway to Assessment and Treatment
    "3bde291790e7": "skin_tear",  # ISTAP Product Selection Guide

    # ── ANZBA (4 chunks) ──────────────────────────────────────────────────────
    "5d08501c9e7a": "burn",       # ANZBA Burn Classification, Depth & Initial Dressing
    "66d4e6fcfa13": "burn",       # ANZBA Referral Criteria
    "73bc744c2269": "burn",       # ANZBA First Aid, Hydrogel, Initial Wound Cover
    "1c099dac6b20": "burn",       # ANZBA Dressing Selection Reference

    # ── RCH (11 chunks) ───────────────────────────────────────────────────────
    "dd7540d2be3e": "paediatric", # TIME Wound Assessment Framework (RCH)
    "24994a09fcdd": "paediatric", # Wound Assessment Infection/Inflammation (RCH)
    "66b18503ea05": "paediatric", # Wound Assessment Moisture/Exudate (RCH)
    "7f0dde09a620": "paediatric", # Factors Affecting Wound Healing (RCH)
    "3d27931cee1d": "paediatric", # Wound Management Principles (RCH)
    "6cff871dda23": "paediatric", # Primary Dressing Selection Table (RCH)
    "69b3535fbd9a": "paediatric", # Secondary Dressing Selection (RCH)
    "dc0189fecaf3": "paediatric", # Acute Traumatic Wound × Dressing Table (RCH)
    "0b0626e45180": "paediatric", # Dressing Choices Full Detail Table (RCH)
    "d2e54ae32ac9": "paediatric", # RCH Product Reference Guide
    "b4b0b902c29a": "paediatric", # RCH Dressings Poster
}


# ── wound_category: chunk_id → category string ────────────────────────────────
# "algorithm"         : wound-type decision algorithm (primary retrieval target)
# "dressing_product"  : per-dressing product reference (WCM Ch14, SFP Table 2)
# "dressing_mechanism": multi-dressing selection logic (WCM Ch14 overview,
#                       SFP categories, AJGP dressing types)
# "wound_specific"    : wound-type-specific assessment + management chapter
# "assessment"        : assessment-focused chunks (TIME, infection, moisture)
# "procedure"         : procedural guidance (debridement, NPWT, pain mgmt)
# "reference"         : glossary, appendix, cost tables

WOUND_CATEGORY_MAP: dict[str, str] = {

    # ── GP ────────────────────────────────────────────────────────────────────
    "8409dedeea26": "assessment",          # GP TIME Framework
    "bd2bb8e1321e": "algorithm",           # GP Decision Algorithm ← primary pinned
    "52ef696853c7": "algorithm",           # GP Wound Type 1
    "4643f10b8894": "algorithm",           # GP Wound Type 2
    "c0a350e36ecf": "algorithm",           # GP Wound Type 3
    "d622ee9f4c9c": "algorithm",           # GP Wound Type 4
    "aad7a40107b0": "algorithm",           # GP Wound Type 5
    "b4ba04cb08d4": "algorithm",           # GP Wound Type 6
    "c4177e98524e": "algorithm",           # GP Wound Type 7
    "e75347f9bdb3": "algorithm",           # GP Wound Type 8
    "ca7a1e934891": "assessment",          # GP Referral criteria
    "8bb5168bd7e5": "assessment",          # GP Tissue type descriptions
    "0cc16688a29a": "reference",           # GP Clinical glossary

    # ── SFP ───────────────────────────────────────────────────────────────────
    "ba05f42e79d0": "reference",
    "08021aac2fa6": "dressing_mechanism",
    "e7ca1b602a48": "dressing_mechanism",
    "18ab673003dc": "dressing_mechanism",
    "07da3fcbedeb": "dressing_mechanism",
    "14535d1dba1a": "dressing_product",
    "8443058d681e": "dressing_product",
    "c42103c45a0c": "dressing_product",
    "80e829def300": "dressing_product",
    "3082e1a296e7": "dressing_product",    # iodine/silver/honey — thyroid contraindication
    "5972b0ef4b66": "dressing_product",
    "b49bc7f134bd": "dressing_product",
    "08bd7cc4c34e": "procedure",
    "9c6c8076a17e": "procedure",
    "138be9123963": "procedure",
    "218879320005": "procedure",
    "52981701d491": "procedure",
    "8b8a70dc7e9e": "procedure",
    "ea9194262a27": "procedure",
    "1fddeefdfb8b": "procedure",
    "a23f76944381": "procedure",
    "352258a65d26": "procedure",
    "8ac10c306384": "reference",
    "25a2969b1f2e": "reference",
    "6a4054e5b255": "reference",
    "b4c13d77818b": "dressing_product",    # Table 2 Hydrocolloids
    "ad036dc35955": "dressing_product",    # Table 2 Hydrogels
    "3b666ccfba99": "dressing_product",    # Table 2 Alginates
    "4b75fefc0517": "dressing_product",    # Table 2 Hydrofiber
    "254dd74d7f00": "dressing_product",    # Table 2 Foams
    "f05456b64b8d": "dressing_product",    # Table 2 Cadexomer Iodine
    "765445bd6358": "dressing_product",    # Table 2 Silver Barrier
    "757123a126d6": "dressing_product",    # Table 2 Non-Adherent
    "9e661711e520": "dressing_product",    # Table 2 Films
    "605598290337": "dressing_product",    # Table 2 Gauze
    "e86bb5a0ee36": "dressing_product",    # Table 2 Composite

    # ── AJGP ──────────────────────────────────────────────────────────────────
    "65414974ff6f": "reference",
    "e33a690e240a": "assessment",
    "2dc1f26b6233": "dressing_mechanism",
    "4b741bf1a512": "dressing_mechanism",
    "6e34c2b50050": "dressing_mechanism",
    "55569f16010f": "wound_specific",      # skin tear table
    "18fdcc9d34d9": "wound_specific",      # minor cut/laceration table
    "6555b7728ccc": "wound_specific",      # postoperative wounds table
    "5116098922da": "wound_specific",      # burns table
    "0d0a9fc09c73": "wound_specific",      # diabetic foot table
    "f7b66b783d4d": "dressing_product",    # Film
    "c9b68759e2a8": "dressing_product",    # Foam
    "e5409b86f782": "dressing_product",    # Low-adherent
    "a1968343e3ba": "dressing_product",    # Hydrocolloid
    "38f855618fe2": "dressing_product",    # Alginate
    "0c49aa8d13c4": "dressing_product",    # Antimicrobial
    "ca472c44f89b": "reference",
    "751c661d8c3c": "reference",
    "10c6b59b1aff": "reference",

    # ── WCM ───────────────────────────────────────────────────────────────────
    "0305af9f4828": "assessment",
    "06f9b3a7dc69": "assessment",
    "9f8aabde769e": "assessment",
    "9eace8705738": "assessment",
    "424af37cbc2e": "assessment",
    "5c0355fe4e4c": "wound_specific",      # Burn chapter
    "594eb6407bf0": "wound_specific",      # Traumatic wound
    "e21d5fa201a5": "wound_specific",      # DFU assessment
    "261f5855cdb7": "wound_specific",      # DFU management
    "4b89a41a249d": "wound_specific",      # Venous ulcer
    "90610861be44": "wound_specific",      # Arterial ulcer
    "cde2bed226ee": "wound_specific",      # Pressure ulcer
    "7ef21910c0c1": "wound_specific",      # Non-healing ulcer
    "c6bd9ad78883": "wound_specific",
    "3ab46ddb1cb9": "wound_specific",      # Necrotizing fasciitis
    "634395d03254": "wound_specific",
    "1101f1aec597": "procedure",
    "42616730845e": "procedure",
    "d1b94d1b0458": "procedure",
    "9177c6a0a15f": "procedure",
    "fd2fb7bfa667": "procedure",
    "fc529d1bd75b": "procedure",           # Ch13 Cleansing
    "1b2820149c46": "dressing_mechanism",  # Ch14 Overview
    "2de03f803f2f": "dressing_product",    # Ch14 Film
    "d81176511903": "dressing_product",    # Ch14 Hydrogel
    "f8cb463d04cf": "dressing_product",    # Ch14 Hydrocolloid
    "c540b3e5c067": "dressing_product",    # Ch14 Alginate
    "77e6e32d188a": "dressing_product",    # Ch14 Foam
    "e63bd0378895": "dressing_product",    # Ch14 Hydrofibre
    "861a57a2172c": "dressing_product",    # Ch14 Charcoal
    "e8c86c4e1aa6": "dressing_product",    # Ch14 Silver
    "6fd9e2433cc9": "dressing_product",    # Ch14 Polymeric Membrane
    "9466f8d45d86": "dressing_product",    # Ch14 Composite
    "ac13950bf23d": "dressing_product",    # Ch14 Other Advanced
    "b5b5a6c9dcf2": "procedure",           # Ch15 Debridement
    "c550f2c4e065": "procedure",
    "c45690235ea6": "procedure",
    "b480aa73a9c2": "dressing_product",    # Ch16a Honey
    "05cc6ca1ddfc": "procedure",           # Ch16c NPWT
    "c799dd10dcf3": "reference",           # Appendix 7 Analgesics

    # ── EWMA ──────────────────────────────────────────────────────────────────
    "34a1a51c74ae": "assessment",          # TIME Framework evolution
    "e905d7d38dad": "algorithm",           # TIME Applied to Practice — second TIME algo source
    "643cd131813b": "assessment",          # TIME Figure 1
    "61346aa382bd": "wound_specific",      # DFU Tissue Management
    "65a59d1430fd": "wound_specific",      # DFU Infection Control
    "9525c0cb50e8": "wound_specific",      # DFU Moisture Balance
    "6231fe39e6d5": "wound_specific",      # DFU Edge Advancement
    "dfb568f55887": "wound_specific",      # DFU Advanced Therapies
    "142adfaa2033": "wound_specific",      # VLU Tissue Management
    "bd87881f1796": "wound_specific",      # VLU Infection Control
    "9d379b10e0c1": "wound_specific",      # VLU Moisture Balance
    "a60e6a06f137": "wound_specific",      # VLU Edge Advancement & Advanced

    # ── ISTAP ─────────────────────────────────────────────────────────────────
    "d2957f5e4841": "assessment",          # ISTAP Classification Types 1, 2, 3
    "c3d5e1f498ba": "algorithm",           # ISTAP Pathway (treatment algorithm)
    "3bde291790e7": "dressing_product",    # ISTAP Product Selection

    # ── ANZBA ─────────────────────────────────────────────────────────────────
    "5d08501c9e7a": "assessment",          # ANZBA Burn Classification & Depth
    "66d4e6fcfa13": "algorithm",           # ANZBA Referral Criteria (algorithmic)
    "73bc744c2269": "procedure",           # ANZBA First Aid
    "1c099dac6b20": "dressing_product",    # ANZBA Dressing Selection Reference

    # ── RCH ───────────────────────────────────────────────────────────────────
    "dd7540d2be3e": "assessment",          # TIME Wound Assessment (RCH)
    "24994a09fcdd": "assessment",          # Infection/Inflammation Assessment (RCH)
    "66b18503ea05": "assessment",          # Moisture/Exudate Assessment (RCH)
    "7f0dde09a620": "assessment",          # Factors Affecting Wound Healing (RCH)
    "3d27931cee1d": "procedure",           # Wound Management Principles (RCH)
    "6cff871dda23": "dressing_product",  # Primary Dressing Selection Table (RCH)
    "69b3535fbd9a": "dressing_product",    # Secondary Dressing Selection (RCH)
    "dc0189fecaf3": "dressing_product",      # Acute Traumatic Wound × Dressing (RCH)
    "0b0626e45180": "dressing_product",    # Dressing Choices Full Detail (RCH)
    "d2e54ae32ac9": "dressing_product",           # Product Reference Guide (RCH)
    "b4b0b902c29a": "dressing_product",    # Dressings Poster (RCH)
}


# ── population: default "adult", override for RCH and other paediatric chunks ─
def _resolve_population(chunk: dict) -> str:
    """
    Determine population field value.
    RCH chunks already have population='paediatric' in their JSON.
    All others default to 'adult'.
    """
    return chunk.get("population", "adult")


print("[Cell 4] Metadata maps defined.")
print(f"  WOUND_TYPE_MAP     : {len(WOUND_TYPE_MAP)} chunk entries")
print(f"  WOUND_CATEGORY_MAP : {len(WOUND_CATEGORY_MAP)} chunk entries")


[Cell 4] Metadata maps defined.
  WOUND_TYPE_MAP     : 138 chunk entries
  WOUND_CATEGORY_MAP : 138 chunk entries


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 5 — Load All 8 JSON Files
# ══════════════════════════════════════════════════════════════════════════════

CHUNK_JSON_FILES = {
    "GP":    "GP_wound_dressings_kept.json",
    "SFP":   "SFP_wound_dressings_kept.json",
    "AJGP":  "AJGP_wound_dressings_kept.json",
    "WCM":   "WCM_wound_care_manual_kept.json",
    "EWMA":  "EWMA_wound_bed_preparation_kept.json",
    "ISTAP": "ISTAP_skin_tear_kept.json",
    "ANZBA": "ANZBA_burns_kept.json",
    "RCH":   "RCH_wound_care_kept.json",
}


def load_all_chunks(chunk_dir: str) -> list[dict]:
    """
    Load chunks from all 8 JSON files.
    Handles both formats:
      - list of dicts (WCM old format)
      - dict with kept_chunks key (GP, SFP, AJGP, EWMA, ISTAP, ANZBA, RCH)
    """
    all_chunks: list[dict] = []
    for src_key, fname in CHUNK_JSON_FILES.items():
        fpath = os.path.join(chunk_dir, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(
                f"Missing file: {fpath}\n"
                f"Set WOUND_CHUNK_DIR env var to the folder containing all 8 kept JSON files."
            )
        raw = json.load(open(fpath, encoding="utf-8"))
        if isinstance(raw, list):
            chunks = raw                         # WCM format
        elif isinstance(raw, dict):
            chunks = raw.get("kept_chunks", [])  # All others
        else:
            raise ValueError(f"Unexpected JSON structure in {fpath}")
        print(f"  ✅ [{src_key:<5}] {len(chunks):>3} chunks loaded from {fname}")
        all_chunks.extend(chunks)
    print(f"\n  Total loaded: {len(all_chunks)} chunks across {len(CHUNK_JSON_FILES)} sources")
    return all_chunks


print("\n[Cell 5] Loading all chunk JSON files...")
all_chunks = load_all_chunks(CHUNK_DIR)



[Cell 5] Loading all chunk JSON files...
  ✅ [GP   ]  13 chunks loaded from GP_wound_dressings_kept.json
  ✅ [SFP  ]  36 chunks loaded from SFP_wound_dressings_kept.json
  ✅ [AJGP ]  19 chunks loaded from AJGP_wound_dressings_kept.json
  ✅ [WCM  ]  40 chunks loaded from WCM_wound_care_manual_kept.json
  ✅ [EWMA ]  12 chunks loaded from EWMA_wound_bed_preparation_kept.json
  ✅ [ISTAP]   3 chunks loaded from ISTAP_skin_tear_kept.json
  ✅ [ANZBA]   4 chunks loaded from ANZBA_burns_kept.json
  ✅ [RCH  ]  11 chunks loaded from RCH_wound_care_kept.json

  Total loaded: 138 chunks across 8 sources


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 6 — chunks_to_documents() — Convert to LangChain Document objects
# ══════════════════════════════════════════════════════════════════════════════

def chunks_to_documents(chunks: list[dict]) -> list[LC_Doc]:
    """
    Convert raw chunk dicts to LangChain Document objects.

    page_content  = ai_summary  (what gets embedded into ChromaDB)
    metadata      = all chunk fields + resolved guideline metadata
                    + wound_type + wound_category + population  [NEW]

    ChromaDB metadata value types must be str, int, float, or bool.
    Lists are serialised to JSON strings.
    None values are converted to empty strings.
    """
    docs: list[LC_Doc] = []
    unknown_wtype    = []
    unknown_wcat     = []
    missing_summary  = []

    for chunk in chunks:
        chunk_id = chunk.get("chunk_id", "UNKNOWN")
        summary  = chunk.get("ai_summary", "").strip()

        if not summary:
            missing_summary.append(chunk_id)
            print(f"  ⚠️  [{chunk_id}] ai_summary is empty — using section as fallback")
            summary = chunk.get("section", chunk.get("title", f"Chunk {chunk_id}"))

        # ── Resolve guideline metadata ─────────────────────────────────────
        g_meta = _resolve_guideline_meta(chunk)

        # ── Resolve NEW metadata fields ────────────────────────────────────
        wound_type = WOUND_TYPE_MAP.get(chunk_id)
        if wound_type is None:
            wound_type = "general"
            unknown_wtype.append(chunk_id)

        wound_category = WOUND_CATEGORY_MAP.get(chunk_id)
        if wound_category is None:
            wound_category = "general"
            unknown_wcat.append(chunk_id)

        population = _resolve_population(chunk)

        # ── Build metadata dict ────────────────────────────────────────────
        # ChromaDB requires flat dict with str/int/float/bool values only.
        metadata = {
            # Core chunk identity
            "chunk_id":       chunk_id,
            "source":         str(chunk.get("source", "")),
            "section":        str(chunk.get("section", "")),
            "parent_section": str(chunk.get("parent_section", "")),
            "chunk_index":    int(chunk.get("chunk_index", 0)),
            "char_count":     int(chunk.get("char_count", len(summary))),

            # Raw text stored separately so generation can reference it
            # without re-embedding (avoids double embedding of formatted text)
            "raw_text":       str(chunk.get("text", summary)),

            # [NEW] Clinical classification metadata
            "wound_type":     wound_type,
            "wound_category": wound_category,
            "population":     population,

            # Guideline authority (from GUIDELINE_METADATA)
            "authority":      str(g_meta.get("authority",      "")),
            "year":           str(g_meta.get("year",           "")),
            "guideline_type": str(g_meta.get("guideline_type", "")),
            "full_name":      str(g_meta.get("full_name",      "")),
            "abbreviation":   str(g_meta.get("abbreviation",   "")),
        }

        # Optional: source_collection (RCH has this field)
        if "source_collection" in chunk:
            metadata["source_collection"] = str(chunk["source_collection"])

        docs.append(LC_Doc(page_content=summary, metadata=metadata))

    # ── Summary warnings ──────────────────────────────────────────────────
    if unknown_wtype:
        print(f"\n  ⚠️  {len(unknown_wtype)} chunks used fallback wound_type='general':")
        for cid in unknown_wtype[:10]:
            print(f"       {cid}")

    if unknown_wcat:
        print(f"\n  ⚠️  {len(unknown_wcat)} chunks used fallback wound_category='general':")
        for cid in unknown_wcat[:10]:
            print(f"       {cid}")

    if missing_summary:
        print(f"\n  ⚠️  {len(missing_summary)} chunks had empty ai_summary (used section fallback):")
        for cid in missing_summary:
            print(f"       {cid}")

    print(f"\n  ✅ Converted {len(docs)} chunks to LangChain Document objects")
    return docs


print("\n[Cell 6] Converting chunks to Document objects...")
documents = chunks_to_documents(all_chunks)




[Cell 6] Converting chunks to Document objects...

  ✅ Converted 138 chunks to LangChain Document objects


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 7 — Ingest into ChromaDB (db_wound_care_v4)
# ══════════════════════════════════════════════════════════════════════════════

def ingest_to_chroma(
    documents:   list[LC_Doc],
    db_dir:      str,
    embed_model: HuggingFaceEmbeddings,
    batch_size:  int = 32,
    overwrite:   bool = True,
) -> Chroma:
    """
    Ingest all documents into ChromaDB.

    overwrite=True (default): removes existing db_dir before ingestion.
    This is the correct behaviour for building v4 from scratch.
    Set overwrite=False only if you want to add to an existing collection.
    """
    db_path = Path(db_dir)

    if overwrite and db_path.exists():
        print(f"  ⚠️  Overwrite=True: removing existing DB at {db_dir}")
        shutil.rmtree(db_path)

    db_path.mkdir(parents=True, exist_ok=True)

    print(f"\n[Cell 7] Ingesting {len(documents)} documents into ChromaDB...")
    print(f"  DB path    : {db_dir}")
    print(f"  Batch size : {batch_size}")

    # Initialise empty collection
    db = Chroma(
        persist_directory=db_dir,
        embedding_function=embed_model,
        collection_metadata={"hnsw:space": "cosine"},
    )

    # Batch upsert to avoid OOM on large collections
    total_batches = (len(documents) + batch_size - 1) // batch_size
    for i in range(0, len(documents), batch_size):
        batch     = documents[i : i + batch_size]
        batch_num = i // batch_size + 1
        ids       = [d.metadata["chunk_id"] for d in batch]

        # Verify all IDs in this batch are unique
        if len(ids) != len(set(ids)):
            dups = [x for x in ids if ids.count(x) > 1]
            raise ValueError(f"Duplicate chunk_ids in batch {batch_num}: {set(dups)}")

        db.add_documents(documents=batch, ids=ids)
        print(f"  Batch {batch_num:>3}/{total_batches}: {len(batch)} docs ingested | "
              f"IDs [{ids[0]} … {ids[-1]}]")

    print(f"\n  ✅ Ingestion complete. {len(documents)} documents in {db_dir}")
    return db


db = ingest_to_chroma(
    documents   = documents,
    db_dir      = DB_DIR,
    embed_model = embedding_model,
    batch_size  = BATCH_SIZE,
    overwrite   = True,     # ← builds db_wound_care_v4 from scratch
)



[Cell 7] Ingesting 138 documents into ChromaDB...
  DB path    : db_wound_care_v4
  Batch size : 32
  Batch   1/5: 32 docs ingested | IDs [8409dedeea26 … ea9194262a27]
  Batch   2/5: 32 docs ingested | IDs [1fddeefdfb8b … 38f855618fe2]
  Batch   3/5: 32 docs ingested | IDs [0c49aa8d13c4 … 77e6e32d188a]
  Batch   4/5: 32 docs ingested | IDs [e63bd0378895 … dd7540d2be3e]
  Batch   5/5: 10 docs ingested | IDs [24994a09fcdd … b4b0b902c29a]

  ✅ Ingestion complete. 138 documents in db_wound_care_v4


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# │ Cell 8 — Verification & Summary
# ══════════════════════════════════════════════════════════════════════════════

print("\n[Cell 8] Verifying ingested DB...")

raw = db.get(include=["metadatas", "documents"])
stored_ids  = raw["ids"]
stored_meta = raw["metadatas"]
stored_docs = raw["documents"]

print(f"\n  Total documents in {DB_DIR}: {len(stored_ids)}")

# ── Breakdown by source abbreviation ─────────────────────────────────────────
source_counts: dict[str, int] = {}
for meta in stored_meta:
    abbr = meta.get("abbreviation", meta.get("source", "UNKNOWN"))
    source_counts[abbr] = source_counts.get(abbr, 0) + 1

print("\n  Documents by source:")
for src, count in sorted(source_counts.items()):
    print(f"    {src:<8}: {count:>3} docs")

# ── Breakdown by wound_type ───────────────────────────────────────────────────
wtype_counts: dict[str, int] = {}
for meta in stored_meta:
    wt = meta.get("wound_type", "MISSING")
    wtype_counts[wt] = wtype_counts.get(wt, 0) + 1

print("\n  Documents by wound_type:")
for wt, count in sorted(wtype_counts.items()):
    print(f"    wound_type={wt:<12}: {count:>3} docs")

# ── Breakdown by wound_category ───────────────────────────────────────────────
wcat_counts: dict[str, int] = {}
for meta in stored_meta:
    wc = meta.get("wound_category", "MISSING")
    wcat_counts[wc] = wcat_counts.get(wc, 0) + 1

print("\n  Documents by wound_category:")
for wc, count in sorted(wcat_counts.items()):
    print(f"    wound_category={wc:<20}: {count:>3} docs")

# ── Breakdown by population ───────────────────────────────────────────────────
pop_counts: dict[str, int] = {}
for meta in stored_meta:
    pop = meta.get("population", "MISSING")
    pop_counts[pop] = pop_counts.get(pop, 0) + 1

print("\n  Documents by population:")
for pop, count in sorted(pop_counts.items()):
    print(f"    population={pop:<12}: {count:>3} docs")

# ── Spot-check: algorithm chunks ─────────────────────────────────────────────
print("\n  Algorithm chunk spot-check (wound_category='algorithm'):")
algo_metas = [m for m in stored_meta if m.get("wound_category") == "algorithm"]
for m in algo_metas:
    print(f"    [{m['chunk_id']}] wtype={m['wound_type']:<12} | abbr={m['abbreviation']:<6} | {m['section'][:60]}")

# ── Spot-check: new source chunks ─────────────────────────────────────────────
print("\n  New source spot-check (EWMA, ISTAP, ANZBA, RCH):")
new_src_abbrs = {"EWMA", "ISTAP", "ANZBA", "RCH"}
new_src_metas = [m for m in stored_meta if m.get("abbreviation") in new_src_abbrs]
for m in new_src_metas[:12]:
    print(
        f"    [{m['chunk_id']}] {m['abbreviation']:<5} | "
        f"wtype={m['wound_type']:<12} | "
        f"wcat={m['wound_category']:<20} | "
        f"pop={m['population']:<12} | "
        f"{m['section'][:50]}"
    )

# ── Smoke test: similarity search ─────────────────────────────────────────────
print("\n  Smoke test — similarity search for 'diabetic foot ulcer dressing':")
test_results = db.similarity_search("diabetic foot ulcer dressing", k=3)
for i, r in enumerate(test_results, 1):
    print(
        f"    [{i}] {r.metadata.get('abbreviation','?'):<6} | "
        f"wtype={r.metadata.get('wound_type','?'):<10} | "
        f"{r.page_content[:80]}..."
    )

print("\n  Smoke test — similarity search for 'skin tear classification silicone foam':")
test_results2 = db.similarity_search("skin tear classification silicone foam", k=3)
for i, r in enumerate(test_results2, 1):
    print(
        f"    [{i}] {r.metadata.get('abbreviation','?'):<6} | "
        f"wtype={r.metadata.get('wound_type','?'):<10} | "
        f"{r.page_content[:80]}..."
    )

print("\n  Smoke test — similarity search for 'burns referral criteria hand face':")
test_results3 = db.similarity_search("burns referral criteria hand face", k=3)
for i, r in enumerate(test_results3, 1):
    print(
        f"    [{i}] {r.metadata.get('abbreviation','?'):<6} | "
        f"wtype={r.metadata.get('wound_type','?'):<10} | "
        f"{r.page_content[:80]}..."
    )

# ── Metadata filter test ──────────────────────────────────────────────────────
print("\n  Metadata filter test — wound_type='1' (should return GP Type 1 chunk):")
try:
    filter_results = db.similarity_search(
        "wound type 1 dressing recommendation clean granulating",
        k=2,
        filter={"wound_type": {"$eq": "1"}},
    )
    for r in filter_results:
        print(
            f"    [{r.metadata.get('chunk_id','?')}] "
            f"wtype={r.metadata.get('wound_type','?')} | "
            f"{r.page_content[:80]}..."
        )
except Exception as e:
    print(f"    Filter test error: {e}")

print("\n  Metadata filter test — population='paediatric' (should return RCH chunks):")
try:
    paed_results = db.similarity_search(
        "paediatric wound dressing",
        k=3,
        filter={"population": {"$eq": "paediatric"}},
    )
    for r in paed_results:
        print(
            f"    [{r.metadata.get('chunk_id','?')}] "
            f"abbr={r.metadata.get('abbreviation','?')} | "
            f"{r.page_content[:80]}..."
        )
except Exception as e:
    print(f"    Filter test error: {e}")

print("\n═══════════════════════════════════════════════════════════")
print(f" db_wound_care_v4 BUILD COMPLETE")
print(f" DB path : {DB_DIR}")
print(f" Total   : {len(stored_ids)} documents")
print("═══════════════════════════════════════════════════════════")
print("\nNext steps:")
print("  1. Run wound_testset_builder_v3.py → wound_testset_v3.json (32 cases)")
print("  2. Run wound_app_v5.py (updated DB path, metadata filters, confidence scoring)")
print("  3. Run RAGAS evaluation (Experiment 2 — KB Expansion)")



[Cell 8] Verifying ingested DB...

  Total documents in db_wound_care_v4: 138

  Documents by source:
            : 108 docs
    ANZBA   :   4 docs
    EWMA    :  12 docs
    ISTAP   :   3 docs
    RCH     :  11 docs

  Documents by wound_type:
    wound_type=1           :   1 docs
    wound_type=2           :   1 docs
    wound_type=3           :   1 docs
    wound_type=4           :   1 docs
    wound_type=5           :   1 docs
    wound_type=6           :   1 docs
    wound_type=7           :   1 docs
    wound_type=8           :   1 docs
    wound_type=burn        :   6 docs
    wound_type=dfu         :   8 docs
    wound_type=general     :  80 docs
    wound_type=paediatric  :  11 docs
    wound_type=procedure   :  16 docs
    wound_type=skin_tear   :   4 docs
    wound_type=vlu         :   5 docs

  Documents by wound_category:
    wound_category=algorithm           :  12 docs
    wound_category=assessment          :  17 docs
    wound_category=dressing_mechanism  :   8 docs
  